# Supply Chain Performance & Risk Analytics
## Phase 1 — Exploratory Data Analysis & Cleaning
**Dataset:** USAID SCMS Delivery History Dataset  
**Rows:** 10,324 | **Columns:** 33  
**Author:** Manohar Choudhary

In [6]:
# import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
print(os.getcwd())


C:\Users\lenovo\OneDrive\Desktop\supply-chain-performance-analytics\python_code


In [2]:
import os

# List everything in data folder
for root, dirs, files in os.walk("../data"):
    for file in files:
        print(os.path.join(root, file))

../data\ 01_eda_cleaning.ipynb
../data\01_eda_cleaning_experimentation.ipynb
../data\.ipynb_checkpoints\ 01_eda_cleaning-checkpoint.ipynb
../data\.ipynb_checkpoints\01_eda_cleaning_experimentation-checkpoint.ipynb
../data\raw_data\SCMS_Delivery_History_Dataset_20150929.csv


In [32]:
# Reading csv file
df = pd.read_csv(
    "../data/raw_data/SCMS_Delivery_History_Dataset_20150929.csv",
    encoding="latin1"
)
print(df.shape)

(10324, 33)


In [10]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (10324, 33)
Columns: ['ID', 'Project Code', 'PQ #', 'PO / SO #', 'ASN/DN #', 'Country', 'Managed By', 'Fulfill Via', 'Vendor INCO Term', 'Shipment Mode', 'PQ First Sent to Client Date', 'PO Sent to Vendor Date', 'Scheduled Delivery Date', 'Delivered to Client Date', 'Delivery Recorded Date', 'Product Group', 'Sub Classification', 'Vendor', 'Item Description', 'Molecule/Test Type', 'Brand', 'Dosage', 'Dosage Form', 'Unit of Measure (Per Pack)', 'Line Item Quantity', 'Line Item Value', 'Pack Price', 'Unit Price', 'Manufacturing Site', 'First Line Designation', 'Weight (Kilograms)', 'Freight Cost (USD)', 'Line Item Insurance (USD)']


## Initial Exploration
Checking data types, null counts, and basic statistics before cleaning.

In [11]:
# datatype and null inspection

print(df.dtypes)
print("\nNull counts:")
print(df.isnull().sum())
print("\nNull percentages:")
print((df.isnull().sum() / len(df)) * 100)

ID                                int64
Project Code                     object
PQ #                             object
PO / SO #                        object
ASN/DN #                         object
Country                          object
Managed By                       object
Fulfill Via                      object
Vendor INCO Term                 object
Shipment Mode                    object
PQ First Sent to Client Date     object
PO Sent to Vendor Date           object
Scheduled Delivery Date          object
Delivered to Client Date         object
Delivery Recorded Date           object
Product Group                    object
Sub Classification               object
Vendor                           object
Item Description                 object
Molecule/Test Type               object
Brand                            object
Dosage                           object
Dosage Form                      object
Unit of Measure (Per Pack)        int64
Line Item Quantity                int64


In [7]:
#Column investigation (Freight Cost, Weight, Shipment Mode)

print("Freight Cost sample values:")
print(df["Freight Cost (USD)"].tail(10))

print("\nWeight sample values:")
print(df["Weight (Kilograms)"].tail(10))

print("\nShipment Mode unique values:")
print(df["Shipment Mode"].value_counts(dropna=False))

Freight Cost sample values:
10314               See DN-4274 (ID#:84472)
10315                                 26180
10316                                  3410
10317               See DN-4282 (ID#:83919)
10318               See DN-4307 (ID#:83920)
10319               See DN-4307 (ID#:83920)
10320               See DN-4313 (ID#:83921)
10321    Freight Included in Commodity Cost
10322    Freight Included in Commodity Cost
10323    Freight Included in Commodity Cost
Name: Freight Cost (USD), dtype: object

Weight sample values:
10314       See DN-4274 (ID#:84472)
10315                         15198
10316                          1547
10317       See DN-4282 (ID#:83919)
10318       See DN-4307 (ID#:83920)
10319       See DN-4307 (ID#:83920)
10320       See DN-4313 (ID#:83921)
10321    Weight Captured Separately
10322                          1392
10323    Weight Captured Separately
Name: Weight (Kilograms), dtype: object

Shipment Mode unique values:
Shipment Mode
Air            6113
Truck

## Data Cleaning
All cleaning steps applied in sequence. Each step documented in issue_log.md

In [43]:
# ════════════════════════════════════════════════════
# SUPPLY CHAIN ANALYTICS — FULL CLEANING PIPELINE
# ════════════════════════════════════════════════════

# ── Issue 008 ── Load data with correct encoding
df = pd.read_csv(
    "../data/raw_data/SCMS_Delivery_History_Dataset_20150929.csv",
    encoding="latin1"
)
print("Raw data loaded. Shape:", df.shape)

# ── Issue 006 ── Freight Type flag (before numeric conversion)
df["Freight_Type"] = np.where(
    df["Freight Cost (USD)"].str.contains(r"[A-Za-z]", na=False),
    "Absorbed", "Separate"
)

# ── Issue 006 ── Convert Freight Cost to numeric
df["Freight Cost (USD)"] = pd.to_numeric(
    df["Freight Cost (USD)"], errors="coerce"
)

# ── Issue 007 ── Weight Type flag (before numeric conversion)
df["Weight_Type"] = np.where(
    df["Weight (Kilograms)"].str.contains(r"[A-Za-z]", na=False),
    "Captured Separately", "Recorded"
)

# ── Issue 007 ── Convert Weight to numeric
df["Weight (Kilograms)"] = pd.to_numeric(
    df["Weight (Kilograms)"], errors="coerce"
)

# ── Issue 005 ── Convert three clean date columns
date_columns = [
    "Scheduled Delivery Date",
    "Delivered to Client Date",
    "Delivery Recorded Date"
]
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce", format="%d-%b-%y")

# ── Issue 009 ── Convert two mixed date columns
mixed_date_columns = [
    "PQ First Sent to Client Date",
    "PO Sent to Vendor Date"
]
for col in mixed_date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce", format="%m/%d/%Y")

# ── Issue 004 ── Fill insurance nulls with 0
df["Line Item Insurance (USD)"] = df["Line Item Insurance (USD)"].fillna(0)

# ── Issue 002 ── Fill Shipment Mode nulls with Unknown
df["Shipment Mode"] = df["Shipment Mode"].fillna("Unknown")

print("Pipeline complete. Shape:", df.shape)
print("\nRemaining nulls:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Raw data loaded. Shape: (10324, 33)
Pipeline complete. Shape: (10324, 35)

Remaining nulls:
PQ First Sent to Client Date    2681
PO Sent to Vendor Date          5732
Dosage                          1736
Weight (Kilograms)              3952
Freight Cost (USD)              4126
dtype: int64


In [44]:
# varification after cleaning

print("Remaining nulls:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nData types after cleaning:")
print(df.dtypes)

print("\nFreight_Type distribution:")
print(df["Freight_Type"].value_counts())

print("\nWeight_Type distribution:")
print(df["Weight_Type"].value_counts())

Remaining nulls:
PQ First Sent to Client Date    2681
PO Sent to Vendor Date          5732
Dosage                          1736
Weight (Kilograms)              3952
Freight Cost (USD)              4126
dtype: int64

Data types after cleaning:
ID                                       int64
Project Code                            object
PQ #                                    object
PO / SO #                               object
ASN/DN #                                object
Country                                 object
Managed By                              object
Fulfill Via                             object
Vendor INCO Term                        object
Shipment Mode                           object
PQ First Sent to Client Date    datetime64[ns]
PO Sent to Vendor Date          datetime64[ns]
Scheduled Delivery Date         datetime64[ns]
Delivered to Client Date        datetime64[ns]
Delivery Recorded Date          datetime64[ns]
Product Group                           object
Sub C

In [16]:
# ── Issue 005 ── Convert all date columns
date_columns = [
    "PQ First Sent to Client Date",
    "PO Sent to Vendor Date",
    "Scheduled Delivery Date",
    "Delivered to Client Date",
    "Delivery Recorded Date"
]
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce", format="%d-%b-%y")

In [38]:
df["PQ First Sent to Client Date"].tail(10)

10314   2014-09-19
10315   2014-09-19
10316   2015-05-04
10317   2015-05-04
10318   2014-10-16
10319   2014-10-16
10320   2014-10-24
10321   2014-08-12
10322   2015-07-01
10323   2014-10-16
Name: PQ First Sent to Client Date, dtype: datetime64[ns]

In [36]:
print(df["Scheduled Delivery Date"].head(10))
print(df["Delivered to Client Date"].head(10))
print(df[date_columns].isnull().sum())


0   2006-06-02
1   2006-11-14
2   2006-08-27
3   2006-09-01
4   2006-08-11
5   2006-09-28
6   2007-01-08
7   2006-11-24
8   2006-12-07
9   2007-01-30
Name: Scheduled Delivery Date, dtype: datetime64[ns]
0   2006-06-02
1   2006-11-14
2   2006-08-27
3   2006-09-01
4   2006-08-11
5   2006-09-28
6   2007-01-08
7   2006-11-24
8   2006-12-07
9   2007-01-30
Name: Delivered to Client Date, dtype: datetime64[ns]
Scheduled Delivery Date     0
Delivered to Client Date    0
Delivery Recorded Date      0
dtype: int64


In [28]:
df2 = pd.read_csv(
    "../data/raw_data/SCMS_Delivery_History_Dataset_20150929.csv",
    encoding="latin1"
)
print(df2["PQ First Sent to Client Date"].head(10))
print(df2["PO Sent to Vendor Date"].head(10))

0    Pre-PQ Process
1    Pre-PQ Process
2    Pre-PQ Process
3    Pre-PQ Process
4    Pre-PQ Process
5    Pre-PQ Process
6    Pre-PQ Process
7    Pre-PQ Process
8    Pre-PQ Process
9    Pre-PQ Process
Name: PQ First Sent to Client Date, dtype: object
0    Date Not Captured
1    Date Not Captured
2    Date Not Captured
3    Date Not Captured
4    Date Not Captured
5    Date Not Captured
6    Date Not Captured
7    Date Not Captured
8    Date Not Captured
9           11/13/2006
Name: PO Sent to Vendor Date, dtype: object


In [35]:
print(df2["PQ First Sent to Client Date"].head(10))
print(df2["PO Sent to Vendor Date"].head(10))

0    Pre-PQ Process
1    Pre-PQ Process
2    Pre-PQ Process
3    Pre-PQ Process
4    Pre-PQ Process
5    Pre-PQ Process
6    Pre-PQ Process
7    Pre-PQ Process
8    Pre-PQ Process
9    Pre-PQ Process
Name: PQ First Sent to Client Date, dtype: object
0    Date Not Captured
1    Date Not Captured
2    Date Not Captured
3    Date Not Captured
4    Date Not Captured
5    Date Not Captured
6    Date Not Captured
7    Date Not Captured
8    Date Not Captured
9           11/13/2006
Name: PO Sent to Vendor Date, dtype: object


In [45]:
# ── Issue 002 ── 3.7 % null value in shipment mode 
df["Shipment Mode"] = df["Shipment Mode"].fillna("Unknown")
print(df["Shipment Mode"].value_counts())

Shipment Mode
Air            6113
Truck          2830
Air Charter     650
Ocean           371
Unknown         360
Name: count, dtype: int64


In [46]:
df["Dosage"].value_counts().head(20)

Dosage
300mg            990
200mg            932
600mg            772
150/300mg        600
150/300/200mg    580
10mg/ml          552
150mg            431
200/50mg         395
300/300mg        301
600/300/300mg    286
150/200/30mg     250
100mg            228
50mg             174
200/300mg        160
80/20mg/ml       158
400mg            156
20mg/ml          152
30mg             144
600/200/300mg    139
150/30mg         133
Name: count, dtype: int64

In [47]:
df["Dosage"].isnull().sum()

np.int64(1736)

In [48]:
print(df[df["Dosage"].isna()]["Product Group"].value_counts())

Product Group
HRDT    1728
MRDT       8
Name: count, dtype: int64


In [49]:
print(df[df["Dosage"].notna()]["Product Group"].value_counts())

Product Group
ARV     8550
ANTM      22
ACT       16
Name: count, dtype: int64


In [50]:
df["Dosage_Applicable"] = np.where(
    df["Product Group"].isin(["HRDT", "MRDT"]),
    "Not Applicable", "Applicable"
)

In [51]:
print(df["Dosage_Applicable"].value_counts())
print(df[["Product Group", "Dosage", "Dosage_Applicable"]].head(20))

Dosage_Applicable
Applicable        8588
Not Applicable    1736
Name: count, dtype: int64
   Product Group     Dosage Dosage_Applicable
0           HRDT        NaN    Not Applicable
1            ARV    10mg/ml        Applicable
2           HRDT        NaN    Not Applicable
3            ARV      150mg        Applicable
4            ARV       30mg        Applicable
5            ARV    10mg/ml        Applicable
6            ARV      200mg        Applicable
7            ARV      200mg        Applicable
8            ARV       30mg        Applicable
9            ARV   200/50mg        Applicable
10           ARV   200/50mg        Applicable
11          HRDT        NaN    Not Applicable
12          HRDT        NaN    Not Applicable
13           ARV  150/300mg        Applicable
14          HRDT        NaN    Not Applicable
15           ARV      200mg        Applicable
16          HRDT        NaN    Not Applicable
17          HRDT        NaN    Not Applicable
18           ARV         2g        A